In [1]:
import os
import re
import pandas as pd
from typing import List, Pattern

# === CONFIGURATION ===
CONFIG_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_8_Test\All_Config_Files"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.1_Config_Files_List_ShallowC.csv"
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

def compile_any(patterns: List[str], flags=re.I | re.M) -> List[Pattern]:
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns: List[Pattern], text: str) -> bool:
    return any(p.search(text) for p in patterns)

# === DEVICE SETUP (CI/YAML & related script files) ===
REAL_DEVICE_PATTERNS = compile_any([
    r'(^|\n)\s*adb\s+devices\b',
    r'(^|\n)\s*adb\s+get-state\b',
    r'(^|\n)\s*adb\s+get-serialno\b',
    r'(^|\n)\s*adb\s+install(\s+-r)?\b',
    r'(^|\n)\s*adb\s+-s\s+\S+\b',
    r'(^|\n)\s*adb\s+shell\b',
    r'(^|\n)\s*adb\s+root\b',
    r'(^|\n)\s*adb\s+shell\s+settings\b',
    r'(^|\n)\s*adb\s+shell\s+input\b',
    r'(^|\n)\s*adb\s+shell\s+pm\s+grant\b',
])

EMULATOR_PATTERNS = compile_any([
    r'(^|\n)\s*(sdkmanager|avdmanager)\b',
    r'(^|\n)\s*emulator\s+(-avd|@)\S+',                # emulator -avd foo | emulator @foo
    r'(^|\n)\s*android-wait-for-emulator\b',
    r'(^|\n)\s*start-emulator\.sh\b',
    # GitHub Actions runners / actions that boot emulators
    r'uses:\s*reactivecircus/android-emulator-runner',
    r'uses:\s*pierotofy/setup-android',
    r'uses:\s*actions/setup-android',
    # Typical YAML keys indicating emulator params
    r'\bapi[-_ ]?level\b\s*:?\s*\d{2}',               # api-level: 34
    r'\babi\b|\barch\b\s*:?\s*(x86|x86_64|arm64|armeabi)',
    r'\btarget\s*:\s*(google_apis|google_apis_playstore|aosp.*)',
    r'\bavd[-_ ]?name\b|\bdevice\s*:\s*pixel',        # device: pixel* / avd-name
])

THIRD_PARTY_PATTERNS = compile_any([
    r'(^|\n)\s*gcloud(\s+beta)?\s+firebase\s+test\s+android\s+run\b',
    r'(^|\n)\s*saucectl(\s+run)?\b',                   # Sauce Labs
    r'(^|\n)\s*(browserstack|bstack)\b',
    r'(^|\n)\s*appcenter\s+test\s+run\s+android\b',
    r'(^|\n)\s*maestro\s+cloud\b',                     # Mobile.dev Maestro Cloud (if present)
    r'\btest_matrix\.json\b|\bfirebase\.json\b',
])

# === INSTRUMENTATION TEST TRIGGERS (Gradle/ADB in CI) ===
INSTRUMENTATION_TRIGGER_PATTERNS = compile_any([
    # Gradle connected tests (module-qualified or not)
    r'(^|\n)\s*(\./)?gradlew(\.bat)?\s+(:[\w-]+:)?connected(android)?test\b',
    r'(^|\n)\s*(\./)?gradlew(\.bat)?\s+(:[\w-]+:)?connected.*android.*test\b',
    r'(^|\n)\s*(\./)?gradlew(\.bat)?\s+(:[\w-]+:)?connectedcheck\b',
    r'(^|\n)\s*(\./)?gradlew(\.bat)?\s+.*createinstrumentationtestcoveragereport\b',
    r'(^|\n)\s*(\./)?gradlew(\.bat)?\s+.*runinstrumentationtests\b',
    r'(^|\n)\s*(\./)?gradlew(\.bat)?\s+.*executescreenshottests\b',
    r'(^|\n)\s*(\./)?gradlew(\.bat)?\s+.*orchestrator\b',
    r'(^|\n)\s*(\./)?gradlew(\.bat)?\s+(:[\w-]+:)?connected.*\b',  # broad, comes last
    # ADB runner
    r'(^|\n)\s*(adb\s+shell\s+)?am\s+instrument\b',
    # Some CI actions provide a "script:" that runs connected tests
    r'\bscript\s*:\s*(\./)?gradlew(\.bat)?\s+(:[\w-]+:)?connected.*',
    # Third-party labs can be both setup and trigger; count as trigger when present
    r'(^|\n)\s*gcloud(\s+beta)?\s+firebase\s+test\s+android\s+run\b',
    r'(^|\n)\s*saucectl(\s+run)?\b',
    r'(^|\n)\s*appcenter\s+test\s+run\s+android\b',
])

# === UNIT TESTS: Trigger (CI) vs Config (Build) ===
UNIT_TEST_TRIGGER_PATTERNS = compile_any([
    # Try to avoid connected/instrumentation by negative lookaheads
    r'(^|\n)\s*(\./)?gradlew(\.bat)?\s+(:[\w-]+:)?test(?!.*connected)(?!.*android)\b',
    r'(^|\n)\s*(\./)?gradlew(\.bat)?\s+(:[\w-]+:)?check(?!.*connected)(?!.*android)\b',
    r'(^|\n)\s*(\./)?gradlew(\.bat)?\s+(:[\w-]+:)?jvmtest\b',
    # build often runs unit tests; keep but de-prioritize false positives via ordering
    r'(^|\n)\s*(\./)?gradlew(\.bat)?\s+(:[\w-]+:)?build(?!.*connected)(?!.*android)\b',
    r'(^|\n)\s*(npm|yarn)\s+test\b',
    r'(^|\n)\s*run:\s*test\b',
])

UNIT_TEST_CONFIG_PATTERNS = compile_any([
    # Gradle config keywords
    r'\btestimplementation\b',
    r'\bkotlin\(["\']test["\']\)',                    # kotlin("test")
    r'\bandroidTestUtil\b',                           # still config-ish
    r'\buseOrchestrator\s*=\s*true\b',                # not unit, but CI config; keep for context
    # Common unit test libs
    r'junit:junit',
    r'org\.junit\.jupiter',
    r'mockito|mockk\b',
    r'\brobolectric\b',
    r'com\.google\.truth:truth',
    r'org\.hamcrest',
])

# === INSTRUMENTATION TEST DEFINITION HINTS (Build)
INSTRUMENT_TEST_CONFIG_PATTERNS = compile_any([
    r'\bandroidtestimplementation\b',
    r'\bandroidandroidtestimplementation\b',           # typo variant
    r'\bandroidtestapi\b',
    r'\bandroidtestcompile\b',
    r'\bandroidx\.test\b',                             # umbrella
    r'test\.ext:junit',
    r'test\.espresso',
    r'test\.uiautomator',
    r'\bespresso-(core|intents|web|contrib)\b',
    r'\buiautomator\b',
    r'\borchestrator\b',
    r'compose\.ui(\.test|-test(-junit4|-manifest)?)',
    r'play-services-test|firebase-testlab',
    r'\btestinstrumentationrunner\b',
    r'\btestinstrumentationrunnerarguments\b',
    r'\bmanageddevices\b|\bdevicegroups\b|\btestoptions\b',
    r'\buseOrchestrator\s*=\s*true\b',
    r'\bandroidTestUtil\b',
])

STRICT_CI_PLATFORMS = ['github_actions', 'gitlab', 'jenkins', 'azure']
LENIENT_CI_PLATFORMS = ['travis_ci', 'bitrise', 'circle_ci', 'appveyor', 'teamcity', 'buddy']

# === MAIN ANALYSIS ===
results = []

for file in os.listdir(CONFIG_DIR):
    file_path = os.path.join(CONFIG_DIR, file)
    if not os.path.isfile(file_path):
        continue

    file_ext = os.path.splitext(file)[-1].lower()
    if file_ext not in ['.sh', '.json', '.gradle', '.kts', '.yml', '.yaml']:
        continue

    full_name, ci_platform = "Unknown", "Unknown"
    if "__" in file and "++" in file:
        try:
            full_name = file.split("__")[0]
            ci_platform = file.split("__")[1].split("++")[0]
        except Exception:
            pass

    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read().lower()

        parsed_ok = True
        file_is_build = file_ext in ['.gradle', '.kts']

        # === INIT VALUES ===
        device_setup = set()
        trigger_detected = False
        test_definition = False
        unit_test_trigger = False
        unit_test_config = False
        instru_test_status = "None"

        if file_is_build:
            # Build files: instrumentation config + unit test config
            if any_match(UNIT_TEST_CONFIG_PATTERNS, content):
                unit_test_config = True
            if any_match(INSTRUMENT_TEST_CONFIG_PATTERNS, content):
                test_definition = True

        else:
            # === DEVICE SETUP (only for CI/YAML/SH/JSON) ===
            if any_match(REAL_DEVICE_PATTERNS, content):
                device_setup.add("Real_Device")
            if any_match(EMULATOR_PATTERNS, content):
                device_setup.add("Emulator")
            if any_match(THIRD_PARTY_PATTERNS, content):
                device_setup.add("Third_Party_Lab")

            # === TEST TRIGGERS ===
            if any_match(INSTRUMENTATION_TRIGGER_PATTERNS, content):
                trigger_detected = True

            # === UNIT TEST TRIGGER (CI) ===
            if any_match(UNIT_TEST_TRIGGER_PATTERNS, content):
                unit_test_trigger = True

            # === STATUS LOGIC (unchanged, improved inputs) ===
            platform_key = ci_platform.strip().lower()
            ds = "None" if not device_setup else ", ".join(sorted(device_setup))
            ht = "Yes" if trigger_detected else "No"

            if file_ext in ['.yml', '.yaml']:
                if ds == "None" and ht == "Yes":
                    if platform_key in STRICT_CI_PLATFORMS:
                        instru_test_status = "Defective"
                    elif platform_key in LENIENT_CI_PLATFORMS:
                        instru_test_status = "Complete"
                elif ds != "None" and ht == "Yes":
                    instru_test_status = "Complete"
                elif ds != "None" and ht == "No":
                    instru_test_status = "Manual"

    except Exception:
        parsed_ok = False
        ds = "None"
        ht = "No"
        instru_test_status = "Error while parsing"

    # === FINALIZE FIELDS FOR EXPORT ===
    results.append({
        'filename': file,
        'file_type': file_ext[1:],
        'full_name': full_name,
        'ci_platform': ci_platform,
        'device_setup': "None" if file_ext in ['.gradle', '.kts'] else (", ".join(sorted(device_setup)) if device_setup else "None"),
        'has_trigger': "No" if file_ext in ['.gradle', '.kts'] else ("Yes" if trigger_detected else "No"),
        'has_test_definition': "Yes" if test_definition else "No",
        'unit_test_trigger': False if file_ext in ['.gradle', '.kts'] else unit_test_trigger,
        'unit_test_config': unit_test_config,
        'instru_test_status': instru_test_status,
        'parsed_ok': parsed_ok
    })

# === EXPORT RESULTS ===
df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Instrumentation analysis complete! Results saved to:\n{OUTPUT_CSV}")


✅ Instrumentation analysis complete! Results saved to:
C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.1_Config_Files_List_ShallowC.csv


In [4]:
import os
import re
import pandas as pd

# === CONFIGURATION ===
CONFIG_DIR = r"C:\Android Mobile App\All_Config_Files"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Instru_Analysis_V2.0\3.1_Config_Files_List_ShallowC.csv"
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

# === DETECTION KEYWORDS ===
REAL_DEVICE_KEYWORDS = [
    'adb devices', 'adb get-state', 'adb get-serialno', 'adb install', 'adb install -r',
    'adb -s', 'adb shell', 'adb root', 'adb shell settings', 'adb shell input', 'adb shell pm grant'
]

EMULATOR_KEYWORDS = [
    'emulator',
    'android-wait-for-emulator',
    'start-emulator.sh',
    'avdmanager create avd',
    'emulator -avd',
    'emulator @',
]

THIRD_PARTY_KEYWORDS = [
    'gcloud firebase test android run', 'browserstack', 'saucectl', 'bstack', 'appcenter test run',
    'test_matrix.json', 'firebase.json'
]

INSTRUMENTATION_TRIGGER_KEYWORDS = [
    'adb shell am instrument', 'am instrument', './gradlew connectedandroidtest',
    'connectedcheck', 'connectedflavortest', 'createinstrumentationtestcoveragereport',
    'runinstrumentationtests', 'executescreenshottests', 'orchestrator', 'connectedtest'
]

UNIT_TEST_KEYWORDS_CI = [
    'gradlew test', './gradlew test', './gradlew jvmtest',
    'testdebugunittest', 'testreleaseunittest', 'kotlintest',
    'unittest', 'run unit tests', 'run: test', 'npm test', 'yarn test'
]

UNIT_TEST_KEYWORDS_BUILD = ['junit', 'testimplementation']
INSTRUMENT_TEST_KEYWORDS_BUILD = ['androidtestimplementation', 'espresso', 'uiautomator']

STRICT_CI_PLATFORMS = ['github_actions', 'gitlab', 'jenkins', 'azure']
LENIENT_CI_PLATFORMS = ['travis_ci', 'bitrise', 'circle_ci', 'appveyor', 'teamcity', 'buddy']

# === HELPERS ===
def unique_hits(keywords, content_lc):
    """Return ordered unique keyword matches (literal substring)."""
    seen = set()
    hits = []
    for k in keywords:
        k_lc = k.lower()
        if k_lc in content_lc and k not in seen:
            seen.add(k)
            hits.append(k)
    return hits

# === MAIN ANALYSIS ===
results = []

for file in os.listdir(CONFIG_DIR):
    file_path = os.path.join(CONFIG_DIR, file)
    if not os.path.isfile(file_path):
        continue

    file_ext = os.path.splitext(file)[-1].lower()
    if file_ext not in ['.sh', '.json', '.gradle', '.kts', '.yml', '.yaml']:
        continue

    full_name, ci_platform = "Unknown", "Unknown"
    if "__" in file and "++" in file:
        try:
            full_name = file.split("__")[0]
            ci_platform = file.split("__")[1].split("++")[0]
        except Exception:
            pass

    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        content_lc = content.lower()

        parsed_ok = True
        file_is_build = file_ext in ['.gradle', '.kts']

        # === INIT VALUES ===
        device_setup = set()
        trigger_detected = False
        trigger_list = []
        test_definition_list = []
        test_definition = False
        unit_test_ci = False
        unit_test_build = False
        instru_test_status = "None"

        # === BUILD FILES: detect test definitions & build unit tests ===
        if file_is_build:
            if any(k in content_lc for k in UNIT_TEST_KEYWORDS_BUILD):
                unit_test_build = True
            # collect instrumentation test definitions
            test_definition_list = unique_hits(INSTRUMENT_TEST_KEYWORDS_BUILD, content_lc)
            if test_definition_list:
                test_definition = True

        else:
            # === DEVICE SETUP ===
            if any(k in content_lc for k in REAL_DEVICE_KEYWORDS):
                device_setup.add("Real_Device")
            if any(k in content_lc for k in EMULATOR_KEYWORDS):
                device_setup.add("Emulator")
            if any(k in content_lc for k in THIRD_PARTY_KEYWORDS):
                device_setup.add("Third_Party_Lab")

            # === TEST TRIGGER(S) ===
            trigger_list = unique_hits(INSTRUMENTATION_TRIGGER_KEYWORDS, content_lc)
            trigger_detected = len(trigger_list) > 0

            # === UNIT TEST (CI) ===
            if any(k in content_lc for k in UNIT_TEST_KEYWORDS_CI):
                unit_test_ci = True

            # === STATUS LOGIC ===
            platform_key = ci_platform.strip().lower()
            ds = "None" if not device_setup else ", ".join(sorted(device_setup))
            ht = "Yes" if trigger_detected else "No"

            if file_ext in ['.yml', '.yaml']:
                if ds == "None" and ht == "Yes":
                    if platform_key in STRICT_CI_PLATFORMS:
                        instru_test_status = "Defective"
                    elif platform_key in LENIENT_CI_PLATFORMS:
                        instru_test_status = "Complete"
                elif ds != "None" and ht == "Yes":
                    instru_test_status = "Complete"
                elif ds != "None" and ht == "No":
                    instru_test_status = "Manual"

    except Exception as e:
        parsed_ok = False
        device_setup = set()
        trigger_detected = False
        trigger_list = []
        test_definition_list = []
        test_definition = False
        unit_test_ci = False
        unit_test_build = False
        instru_test_status = "Error while parsing"

    # --- finalize fields derived from sets/lists ---
    device_setup_str = "None" if not device_setup or file_is_build else ", ".join(sorted(device_setup))
    has_device_setup = "Yes" if device_setup_str != "None" else "No"

    test_trigger_str = "none" if file_is_build or not trigger_list else ", ".join(trigger_list)
    test_definition_str = "none" if not test_definition_list else ", ".join(test_definition_list)

    # === FINALIZE FIELDS FOR EXPORT ===
    results.append({
        'filename': file,
        'file_type': file_ext[1:],
        'full_name': full_name,
        'ci_platform': ci_platform,
        'has_test_definition': "Yes" if test_definition else "No",
        'has_device_setup': has_device_setup,          # (1) NEW
        'has_trigger': "No" if file_is_build else ("Yes" if trigger_detected else "No"),
        'test_definition': test_definition_str,         # (3) NEW
        'device_setup': device_setup_str,
        'test_trigger': test_trigger_str,               # (2) NEW
        'unit_test_ci': False if file_is_build else unit_test_ci,
        'unit_test_build': unit_test_build,
        'instru_test_status': instru_test_status

    })

# === EXPORT RESULTS ===
df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Instrumentation analysis complete! Results saved to:\n{OUTPUT_CSV}")


✅ Instrumentation analysis complete! Results saved to:
C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\July_31\Instru_Analysis_V2.0\3.1_Config_Files_List_ShallowC.csv
